# Chittagonian ↔ Bengali Neural Machine Translation

## Notebook 03

### mBART-50 Fine-tuning

**Translation Direction**

Standard Bengali → Chatgaya

---

Author : Mostafa Al Moin

Model : facebook/mbart-large-50-many-to-many-mmt

Dataset : 1800 Manual Sentence Pairs

In [ ]:
!pip install -q \
transformers==4.46.3 \
datasets==3.1.0 \
accelerate==1.1.1 \
evaluate==0.4.3 \
sentencepiece==0.2.0 \
sacrebleu==2.4.3 \
peft==0.13.2 \
huggingface-hub==0.26.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 11.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.38.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.26.2 which is incompatible.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.26.2 which is incompatible.


In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
import transformers
import datasets
import accelerate
import evaluate
import peft

print("Transformers :", transformers.__version__)
print("Datasets     :", datasets.__version__)
print("Accelerate   :", accelerate.__version__)
print("Evaluate     :", evaluate.__version__)
print("PEFT         :", peft.__version__)

Transformers : 4.46.3
Datasets     : 3.1.0
Accelerate   : 1.1.1
Evaluate     : 0.4.3
PEFT         : 0.13.2


In [ ]:
import torch

print("=" * 60)

print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

print("=" * 60)

CUDA Available : True
GPU : Tesla T4


In [ ]:
import random
import numpy as np
import pandas as pd

from datasets import Dataset

from transformers import (
    MBart50TokenizerFast,
    MBartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed Fixed :", SEED)

Seed Fixed : 42


In [ ]:
train_df = pd.read_csv("train.csv")
valid_df = pd.read_csv("valid.csv")

print("="*60)

print("Train Shape :", train_df.shape)
print("Valid Shape :", valid_df.shape)

print("="*60)

display(train_df.head())

Train Shape : (1440, 2)
Valid Shape : (180, 2)


,bangla,chatgaya
0,"তোমাদেরকে একটা কাজ দিয়েছিলাম, ভুলে গেছো তোমরা","তোঁয়ারারে এক্কান হাম দিইলাম, ভুলি গেইয়্যু তোঁয়ারা"
1,আমার সবচেয়ে খারাপ সময়ে আমার কোনো বন্ধু পাশে ...,আঁর বিয়াজ্ঞুনুত্তুন হারাপ সময়ুত আঁর হনো বন্ধু ...
2,"টুরে যাওয়ার প্ল্যান করলে সবাই যাবে বলে, কিন্তু...","টুরুত যাইবের প্ল্যান গরিলি বিয়াজ্ঞুন যাইবু হয়,..."
3,আমি সাইকেল চালাতে পারি কিন্তু বাইক চালাতে পারি না,আঁই সাইকেল চালাইত ফারি কিন্তু বাইক চালাইত ন ফারি
4,এটা কি বাসার দারোয়ান?,ইবে কি বাসার দারোওয়ান?


In [ ]:
print(train_df.columns)

print()

print(train_df.isnull().sum())

print()

print(valid_df.isnull().sum())

Index(['bangla', 'chatgaya'], dtype='object')

bangla      0
chatgaya    0
dtype: int64

bangla      0
chatgaya    0
dtype: int64


In [ ]:
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)

print(train_dataset)
print(valid_dataset)

Dataset({
    features: ['bangla', 'chatgaya'],
    num_rows: 1440
})
Dataset({
    features: ['bangla', 'chatgaya'],
    num_rows: 180
})


In [ ]:
# ==========================================================
# Load mBART Tokenizer
# ==========================================================

MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)

print("Tokenizer Loaded Successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Tokenizer Loaded Successfully!


In [ ]:
# ==========================================================
# Language Configuration
# ==========================================================

tokenizer.src_lang = "bn_IN"
tokenizer.tgt_lang = "bn_IN"

print("Source Language :", tokenizer.src_lang)
print("Target Language :", tokenizer.tgt_lang)

Source Language : bn_IN
Target Language : bn_IN


In [ ]:
# ==========================================================
# Load mBART Model
# ==========================================================

model = MBartForConditionalGeneration.from_pretrained(MODEL_NAME)

print("Model Loaded Successfully!")

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Model Loaded Successfully!


In [ ]:
# ==========================================================
# Configuration
# ==========================================================

MAX_SOURCE_LENGTH = 64
MAX_TARGET_LENGTH = 64

print("Max Source Length :", MAX_SOURCE_LENGTH)
print("Max Target Length :", MAX_TARGET_LENGTH)

Max Source Length : 64
Max Target Length : 64


In [ ]:
# ==========================================================
# Preprocessing Function
# ==========================================================

def preprocess_function(examples):

    model_inputs = tokenizer(
        examples["bangla"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=examples["chatgaya"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [ ]:
# ==========================================================
# Tokenize Dataset
# ==========================================================

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_valid = valid_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=valid_dataset.column_names
)

print(tokenized_train)
print(tokenized_valid)

Map:   0%|          | 0/1440 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1440
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 180
})


In [ ]:
# ==========================================================
# Verify Tokenization
# ==========================================================

sample = tokenized_train[0]

print(sample.keys())

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [ ]:
# ==========================================================
# Decode Source Sentence
# ==========================================================

print(tokenizer.decode(sample["input_ids"], skip_special_tokens=True))

তোমাদেরকে একটা কাজ দিয়েছিলাম, ভুলে গেছো তোমরা


In [ ]:
# ==========================================================
# Decode Target Sentence
# ==========================================================

print(tokenizer.decode(sample["labels"], skip_special_tokens=True))

তোঁয়ারারে এক্কান হাম দিইলাম, ভুলি গেইয়্যু তোঁয়ারা


In [ ]:
# ==========================================================
# Data Collator
# ==========================================================

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

print("Data Collator Ready!")

Data Collator Ready!


In [ ]:
# ==========================================================
# Training Arguments
# ==========================================================

training_args = Seq2SeqTrainingArguments(

    output_dir="./mbart_bn2ctg",

    num_train_epochs=10,

    learning_rate=5e-5,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    gradient_accumulation_steps=2,

    evaluation_strategy="epoch",

    save_strategy="epoch",

    logging_strategy="steps",

    logging_steps=25,

    save_total_limit=2,

    predict_with_generate=True,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False,

    fp16=torch.cuda.is_available(),

    report_to="none",

    seed=42
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# ==========================================================
# Trainer
# ==========================================================

trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_train,

    eval_dataset=tokenized_valid,

    tokenizer=tokenizer,

    data_collator=data_collator
)

print("Trainer Ready!")

/tmp/ipykernel_2365/1158737829.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Trainer Ready!


In [ ]:
# ==========================================================
# Start Training
# ==========================================================

trainer.train()

Epoch,Training Loss,Validation Loss
1,1.306100,1.196230
2,0.825400,0.826149
3,0.422500,0.741795
4,0.273800,0.706187
5,0.174300,0.723202
6,0.095000,0.748667
7,0.060400,0.772483
8,0.035000,0.761069
9,0.021500,0.765523
10,0.017400,0.759558


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:2817: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=1800, training_loss=0.42879908134539924, metrics={'train_runtime': 3084.2441, 'train_samples_per_second': 4.669, 'train_steps_per_second': 0.584, 'total_flos': 661110024830976.0, 'train_loss': 0.42879908134539924, 'epoch': 10.0})

In [ ]:
# ==========================================================
# Save Best Model
# ==========================================================

SAVE_PATH = "mbart_bn2ctg_best"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("✅ Model Saved Successfully!")

✅ Model Saved Successfully!


In [ ]:
import os

print(os.listdir("mbart_bn2ctg_best"))

['model.safetensors', 'special_tokens_map.json', 'tokenizer_config.json', 'generation_config.json', 'sentencepiece.bpe.model', 'tokenizer.json', 'config.json', 'training_args.bin']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil

DESTINATION = "/content/drive/MyDrive/Chatgaya-Bangla-NMT/mbart_bn2ctg_best"

shutil.copytree(
    "mbart_bn2ctg_best",
    DESTINATION,
    dirs_exist_ok=True
)

print("✅ Model Backed Up to Google Drive")

✅ Model Backed Up to Google Drive
